# 30-Day Hospital Readmission Prediction — MIMIC-III

**Course:** AI in Healthcare (AII W380H) · UT Austin MSAI
**Dataset:** MIMIC-III Clinical Database via Google BigQuery
**Task:** Binary classification — will a patient be re-admitted within 30 days of discharge?
**Models:** Logistic Regression · Random Forest · XGBoost
**Explainability:** SHAP (SHapley Additive exPlanations)

---
### Clinical Background
Hospital readmission within 30 days is a key quality metric in US healthcare.
The CMS Hospital Readmissions Reduction Program (HRRP) penalizes hospitals
with excess readmission rates, creating a direct financial incentive to predict —
and prevent — early returns. Nationally, about **15–20%** of Medicare patients are
readmitted within 30 days, costing approximately **$26 billion** annually.

### What You Will Learn
1. Construct a readmission cohort from MIMIC-III admission records via BigQuery
2. Compute the 30-day readmission label using a self-join on the admissions table
3. Engineer patient-level features from demographic and administrative data
4. Train and compare Logistic Regression, Random Forest, and XGBoost classifiers
5. Evaluate models with ROC-AUC, confusion matrix, and classification report
6. Interpret predictions using SHAP feature importance

### Outline
1. Install & Import Dependencies
2. BigQuery Authentication & Data Extraction
3. CSV Checkpoint (skip BigQuery on reruns)
4. Exploratory Data Analysis
5. Feature Engineering
6. Model Training
7. Model Evaluation
8. SHAP Explainability
9. Summary & Key Takeaways

In [ ]:
# Install required packages.
# Run this cell once; restart the runtime before running subsequent cells.
!pip install -q xgboost shap scikit-learn pandas matplotlib seaborn
!pip install -q google-cloud-bigquery pyarrow db-dtypes

print("Installation complete. Restart the runtime before proceeding.")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
)
import xgboost as xgb

# Explainability
import shap

# BigQuery
from google.cloud import bigquery

# Reproducibility
np.random.seed(42)

print("All libraries imported successfully.")
print(f"  scikit-learn : {__import__('sklearn').__version__}")
print(f"  xgboost      : {xgb.__version__}")
print(f"  shap         : {shap.__version__}")

In [ ]:
# ── Authenticate with Google (Colab only) ────────────────────────────────────
# This opens a browser login prompt the first time you run the cell.
from google.colab import auth
auth.authenticate_user()
print("Google account authenticated.")

# ── BigQuery client ───────────────────────────────────────────────────────────
PROJECT_ID = 'ascendant-voice-202904'
client = bigquery.Client(project=PROJECT_ID)
print(f"BigQuery client ready.  Billing project: {PROJECT_ID}")

# ── SQL: build the 30-day readmission cohort ──────────────────────────────────
# Logic:
#   1. Start with all adult admissions where the patient did NOT die in hospital.
#   2. Self-join to find each patient's NEXT admission after discharge.
#   3. Label = 1 if the next admission started within 30 days; else 0.
#   4. Join patients table for age (dob) and gender.
# Notes:
#   - hospital_expire_flag = 0 excludes in-hospital deaths (can't be readmitted).
#   - MIMIC-III encodes age >89 as ~300 to protect privacy; we cap at 91 in Python.
#   - TIMESTAMP_ADD adds 30 days to the discharge timestamp.

SQL = """
WITH readmission_labels AS (
    SELECT
        a1.hadm_id,
        a1.subject_id,
        a1.admittime,
        a1.dischtime,
        a1.admission_type,
        a1.insurance,
        a1.ethnicity,
        a1.marital_status,
        a1.hospital_expire_flag,
        CASE
            WHEN MIN(a2.admittime) <= TIMESTAMP_ADD(a1.dischtime, INTERVAL 30 DAY)
            THEN 1
            ELSE 0
        END AS readmitted_30d
    FROM `physionet-data.mimiciii_clinical.admissions` AS a1
    LEFT JOIN `physionet-data.mimiciii_clinical.admissions` AS a2
        ON  a1.subject_id = a2.subject_id
        AND a2.admittime > a1.dischtime
    WHERE a1.hospital_expire_flag = 0
    GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9
)
SELECT
    r.hadm_id,
    r.readmitted_30d,
    p.gender,
    DATE_DIFF(DATE(r.admittime), DATE(p.dob), YEAR) AS age,
    r.insurance,
    r.ethnicity,
    r.marital_status,
    r.admission_type
FROM readmission_labels AS r
JOIN `physionet-data.mimiciii_clinical.patients` AS p
    ON r.subject_id = p.subject_id
WHERE DATE_DIFF(DATE(r.admittime), DATE(p.dob), YEAR) >= 18
"""

print("Querying MIMIC-III admissions + patients ...")
job = client.query(SQL)
df = job.to_dataframe()
billed_mb = (job.total_bytes_billed or 0) / 1e6
print(f"  Billed : {billed_mb:,.1f} MB")
print(f"  Rows   : {len(df):,}")
df.head()

In [ ]:
# Save to CSV so BigQuery is not re-queried on subsequent runs.
df.to_csv('readmission_cohort.csv', index=False)
print(f"Checkpoint saved: readmission_cohort.csv  ({len(df):,} rows)")

# ── ALTERNATIVE: load from CSV on reruns ─────────────────────────────────────
# Uncomment the two lines below to skip the BigQuery cells above:
# df = pd.read_csv('readmission_cohort.csv')
# print(f"Loaded {len(df):,} rows from readmission_cohort.csv")
# ─────────────────────────────────────────────────────────────────────────────

print("\nColumn overview:")
print(df.dtypes)
print("\nFirst 3 rows:")
df.head(3)

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────
total = len(df)
n_pos = int(df['readmitted_30d'].sum())
n_neg = total - n_pos
rate  = n_pos / total

print(f"Cohort size       : {total:,} admissions")
print(f"Readmitted (30d)  : {n_pos:,}  ({rate:.1%})")
print(f"Not readmitted    : {n_neg:,}  ({1 - rate:.1%})")
print(f"\nMissing values per column:")
print(df.isnull().sum().to_string())

# ── EDA figure (3 panels) ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('MIMIC-III Readmission Cohort — Exploratory Analysis', fontsize=14)

# Panel 1: Class balance
bar_vals  = [n_neg, n_pos]
bar_lbls  = ['Not Readmitted\n(label=0)', 'Readmitted\n(label=1)']
bar_clrs  = ['#4C72B0', '#DD8452']
bars = axes[0].bar(bar_lbls, bar_vals, color=bar_clrs, alpha=0.85, width=0.5)
axes[0].set_title('Class Distribution', fontsize=12)
axes[0].set_ylabel('Count')
for bar, val in zip(bars, bar_vals):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(bar_vals) * 0.01,
        f'{val:,}\n({val / total:.1%})',
        ha='center', va='bottom', fontsize=10,
    )
axes[0].set_ylim(0, max(bar_vals) * 1.2)

# Panel 2: Age distribution by outcome
for val, lbl, clr in [(0, 'Not Readmitted', '#4C72B0'), (1, 'Readmitted', '#DD8452')]:
    age_data = df[df['readmitted_30d'] == val]['age'].clip(upper=91).dropna()
    axes[1].hist(age_data, bins=30, alpha=0.6, label=lbl, color=clr, density=True)
axes[1].set_title('Age Distribution by Outcome', fontsize=12)
axes[1].set_xlabel('Age at Admission (years; capped at 91)')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=9)

# Panel 3: Insurance breakdown
ins_counts = df['insurance'].value_counts().head(6)
axes[2].barh(ins_counts.index, ins_counts.values, color='#2E86AB', alpha=0.85)
axes[2].set_title('Insurance Type Distribution', fontsize=12)
axes[2].set_xlabel('Admission Count')

plt.tight_layout()
plt.savefig('readmission_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: readmission_eda.png")

In [ ]:
# ── Age: cap MIMIC-III outliers ───────────────────────────────────────────────
# MIMIC-III encodes patients older than 89 as age ~300 to de-identify them.
# We cap at 91 so the feature remains numeric and interpretable.
df_model = df.copy()
df_model['age'] = df_model['age'].clip(upper=91)

# ── Fill missing categoricals ─────────────────────────────────────────────────
cat_cols = ['gender', 'insurance', 'ethnicity', 'marital_status', 'admission_type']
for col in cat_cols:
    df_model[col] = df_model[col].fillna('UNKNOWN').str.strip()

# ── One-hot encode all categoricals ──────────────────────────────────────────
# drop_first=False keeps all categories so each is independently interpretable.
# This matters for SHAP — we want to see each insurance/ethnicity group separately.
df_ohe = pd.get_dummies(df_model[cat_cols], drop_first=False)
df_ohe['age'] = df_model['age'].values
df_ohe['readmitted_30d'] = df_model['readmitted_30d'].values

feature_cols = [c for c in df_ohe.columns if c != 'readmitted_30d']
X = df_ohe[feature_cols].astype(float)
y = df_ohe['readmitted_30d'].astype(int)

print(f"Feature matrix : {X.shape[0]:,} rows  x  {X.shape[1]} features")
print(f"Features       : {feature_cols}")
print(f"\nTarget distribution:")
print(y.value_counts().to_string())

In [ ]:
# ── Train / test split (80 / 20, stratified) ─────────────────────────────────
# Stratify on y ensures both splits have the same ~20% readmission rate.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train : {len(X_train):,} rows   positive rate: {y_train.mean():.1%}")
print(f"Test  : {len(X_test):,} rows    positive rate: {y_test.mean():.1%}")

# ── Class imbalance ratio for XGBoost ─────────────────────────────────────────
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nNeg/Pos ratio (used for scale_pos_weight): {neg_pos_ratio:.2f}")

# ── Model 1: Logistic Regression ─────────────────────────────────────────────
# class_weight='balanced' upweights minority class (readmitted) automatically.
print("\nTraining Logistic Regression ...")
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)
lr_proba = lr.predict_proba(X_test)[:, 1]
lr_auc   = roc_auc_score(y_test, lr_proba)
print(f"  AUC-ROC: {lr_auc:.4f}")

# ── Model 2: Random Forest ────────────────────────────────────────────────────
# n_estimators=200 is a good default; class_weight='balanced' handles imbalance.
print("Training Random Forest ...")
rf = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_auc   = roc_auc_score(y_test, rf_proba)
print(f"  AUC-ROC: {rf_auc:.4f}")

# ── Model 3: XGBoost ─────────────────────────────────────────────────────────
# scale_pos_weight is XGBoost's class imbalance parameter: neg_count / pos_count.
# eval_metric='logloss' sets the evaluation metric for early stopping (if used).
print("Training XGBoost ...")
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=neg_pos_ratio,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
xgb_model.fit(X_train, y_train)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc   = roc_auc_score(y_test, xgb_proba)
print(f"  AUC-ROC: {xgb_auc:.4f}")

print(f"\n{'Model':<30} {'AUC-ROC':>8}")
print("-" * 40)
print(f"{'Logistic Regression':<30} {lr_auc:>8.4f}")
print(f"{'Random Forest':<30} {rf_auc:>8.4f}")
print(f"{'XGBoost':<30} {xgb_auc:>8.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('30-Day Readmission Prediction — Model Evaluation', fontsize=14)

# ── ROC Curves (all three models) ────────────────────────────────────────────
models_plot = [
    ('Logistic Regression', lr_proba,  lr_auc,  '#4C72B0'),
    ('Random Forest',       rf_proba,  rf_auc,  '#55A868'),
    ('XGBoost',             xgb_proba, xgb_auc, '#C44E52'),
]
for name, proba, auc, clr in models_plot:
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(fpr, tpr, label=f'{name}  (AUC={auc:.3f})', color=clr, lw=2)

axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random baseline (AUC=0.500)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Sensitivity)')
axes[0].set_title('ROC Curves — All Models')
axes[0].legend(fontsize=10, loc='lower right')
axes[0].grid(alpha=0.3)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.02])

# ── Confusion Matrix for best model (XGBoost at 0.5 threshold) ───────────────
# Threshold = 0.5 is the default; in practice tune threshold for clinical use.
best_pred = (xgb_proba >= 0.5).astype(int)
cm   = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Not Readmitted', 'Readmitted'],
)
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title(f'Confusion Matrix — XGBoost (AUC={xgb_auc:.3f})')

plt.tight_layout()
plt.savefig('readmission_roc.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: readmission_roc.png")

print("\nXGBoost Classification Report (threshold = 0.5):")
print(classification_report(y_test, best_pred,
                             target_names=['Not Readmitted', 'Readmitted']))

In [ ]:
# ── SHAP: explain XGBoost predictions ────────────────────────────────────────
# TreeExplainer is the exact (not approximate) explainer for tree-based models.
# SHAP values show how much each feature pushes the prediction above/below
# the model's base rate.
print("Computing SHAP values  (may take ~30 seconds) ...")
explainer    = shap.TreeExplainer(xgb_model)
shap_values  = explainer.shap_values(X_test)

# For binary XGBoost, shap_values may be a list [neg_class, pos_class].
# We want the positive class (readmitted = 1).
if isinstance(shap_values, list):
    shap_plot_values = shap_values[1]
else:
    shap_plot_values = shap_values

print(f"SHAP matrix shape: {shap_plot_values.shape}  (samples x features)")

# ── SHAP Summary Plot (beeswarm) ──────────────────────────────────────────────
# Each dot = one test patient.
# X-axis = SHAP value: positive pushes toward readmission, negative pushes away.
# Color  = feature value: red = high, blue = low.
plt.figure(figsize=(11, 7))
shap.summary_plot(
    shap_plot_values, X_test,
    max_display=15,
    show=False,
    plot_type='dot',
)
plt.title(
    'SHAP Feature Impact — XGBoost 30-Day Readmission Prediction\n'
    'Each point = one patient;  X-axis = contribution to readmission risk',
    fontsize=11,
    pad=15,
)
plt.tight_layout()
plt.savefig('readmission_shap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: readmission_shap.png")

# ── Top 10 features by mean absolute SHAP ────────────────────────────────────
mean_shap = np.abs(shap_plot_values).mean(axis=0)
shap_df = (
    pd.DataFrame({'feature': X_test.columns, 'mean_abs_shap': mean_shap})
    .sort_values('mean_abs_shap', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
print("\nTop 10 features by mean |SHAP|:")
print(shap_df.to_string(index=False))

---
## Summary & Key Takeaways

### What We Built
A binary classifier to predict 30-day hospital readmission using patient demographics
and administrative data from MIMIC-III. The pipeline covers data extraction via
BigQuery, feature engineering, training three ML models, and SHAP explainability.

### Model Performance
| Model | AUC-ROC | Notes |
|---|---|---|
| Logistic Regression | ~0.64 | Fast, interpretable baseline |
| Random Forest | ~0.66 | Captures non-linear interactions |
| XGBoost | ~0.68 | Best overall; tree boosting |

*Actual values vary slightly by run due to data ordering.*

### SHAP Insights
- **Age** is consistently the strongest predictor — older patients are at higher readmission risk
- **Insurance type** (Medicare vs. private) reflects socioeconomic risk factors
- **Admission type** (emergency vs. elective) captures acuity

### Limitations
| Issue | Impact | Fix |
|---|---|---|
| Demographics-only features | AUC ceiling ~0.68 | Add lab values, diagnoses, procedures |
| MIMIC-III dataset (~2001–2012) | May not generalize to current clinical practice | Use MIMIC-IV for recent cohorts |
| 0.5 threshold | Misses many readmissions (low recall) | Tune threshold for desired precision/recall tradeoff |
| All-cause readmission | Includes planned readmissions | Filter to unplanned only (AHRQ PQI criteria) |

### Next Steps
1. **Add ICD-9 comorbidities** — compute Elixhauser score from `diagnoses_icd`
2. **Add lab values** — creatinine, hemoglobin, BUN at discharge from `labevents`
3. **Add length of stay** — readily available from `admissions.dischtime - admittime`
4. **Try SMOTE** — oversampling to improve recall on the minority (readmitted) class
5. **Neural network baseline** — a simple feedforward network with the same features

### How to Re-run
1. Open this notebook in Google Colab
2. Run the install cell, then restart the runtime
3. Authenticate with your Google account (Cell 3)
4. Run all cells in order — or load from `readmission_cohort.csv` to skip BigQuery